# SumCheck Deep Dive: I/O Contract, Operation Taxonomy, and Hardware Execution

> **ECE-9413 Assignment 2 — Companion Analysis Notebook**  
> Covers three things the tutorial notebook doesn't spell out explicitly:
> 1. The **exact I/O contract** `sumcheck_32` must satisfy (dtypes, shapes, values — with worked examples)
> 2. A **taxonomy of every operation** in the algorithm (vec⊙vec, vec⊙scalar, reduction, etc.)
> 3. **How CPUs execute it today** vs **how GPUs/TPUs can execute it** — and why the gap is large

---

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)

print("JAX version :", jax.__version__)
print("Backend     :", jax.default_backend())
print("Devices     :", jax.devices())

JAX version : 0.7.2
Backend     : gpu
Devices     : [CudaDevice(id=0)]


---
## Section 1 — The SumCheck I/O Contract

### What the harness calls

```python
claim0, round_evals = sumcheck_32(
    eval_tables,          # dict[str → jnp.ndarray]
    q          = q,       # uint32 scalar
    expression = expr,    # list[list[str]]
    challenges = chals,   # jnp.ndarray uint32, length = num_rounds - 1
    num_rounds = n,       # int
)
```

Every parameter has a **precise contract**. Getting any one of them wrong — wrong dtype, wrong shape, off-by-one on challenges — produces silent numerical errors that the verifier will catch.

---
### 1.1 — Input: `eval_tables`

| Property | Value |
|---|---|
| Type | `dict[str, jnp.ndarray]` |
| Keys | Variable names from `VARIABLE_NAMES = ('a','b','c','d','e','g')` — only the vars that appear in `expression` need to be present |
| Value dtype | `jnp.uint32` |
| Value shape | `(2**num_rounds,)` — a flat 1-D array of length exactly $2^n$ |
| Value range | Every element is in `[0, q-1]` — already reduced mod `q` |

**Bit-ordering convention (critical):**  
Index `i` into the table corresponds to the Boolean assignment where **bit 0 (LSB) of `i` is `x1`**, bit 1 is `x2`, etc.  
So for `num_rounds=3`:

```
index 0 = 0b000  →  x3=0, x2=0, x1=0
index 1 = 0b001  →  x3=0, x2=0, x1=1
index 2 = 0b010  →  x3=0, x2=1, x1=0
index 3 = 0b011  →  x3=0, x2=1, x1=1
index 4 = 0b100  →  x3=1, x2=0, x1=0
...
index 7 = 0b111  →  x3=1, x2=1, x1=1
```

The MLE update in round `i` always folds on **x1** (the current LSB), which is why even/odd splitting (`table[0::2]` and `table[1::2]`) is the right operation.

In [2]:
# ── Concrete example: eval_tables for a 3-variable problem ──────────────────
Q = jnp.uint32(17)        # tiny prime for readability
num_rounds = 3             # 3 Boolean variables  →  2^3 = 8 entries per table

# Two variables: a and b
# a(x1,x2,x3) = x1 + 2*x2 + 4*x3   (evaluated at all 8 Boolean points, mod 17)
# b(x1,x2,x3) = 1 - x1              (a simple complement)
eval_tables = {
    'a': jnp.array([0, 1, 2, 3, 4, 5, 6, 7], dtype=jnp.uint32),
    'b': jnp.array([1, 0, 1, 0, 1, 0, 1, 0], dtype=jnp.uint32),
}

print("eval_tables contract check")
print("=" * 50)
for name, tbl in eval_tables.items():
    print(f"  '{name}':  dtype={tbl.dtype}  shape={tbl.shape}  "
          f"range=[{int(tbl.min())}, {int(tbl.max())}]")
    assert tbl.dtype == jnp.uint32, "Must be uint32"
    assert tbl.shape == (2**num_rounds,), f"Shape must be (2^{num_rounds},)"
    assert int(tbl.max()) < 17, "All values must be < q"

print("\nIndex→assignment mapping for 'a':")
for i, v in enumerate(eval_tables['a'].tolist()):
    x1 = (i >> 0) & 1
    x2 = (i >> 1) & 1
    x3 = (i >> 2) & 1
    print(f"  index {i} = 0b{i:03b}  (x3={x3}, x2={x2}, x1={x1})  a={v}")

eval_tables contract check
  'a':  dtype=uint32  shape=(8,)  range=[0, 7]
  'b':  dtype=uint32  shape=(8,)  range=[0, 1]

Index→assignment mapping for 'a':
  index 0 = 0b000  (x3=0, x2=0, x1=0)  a=0
  index 1 = 0b001  (x3=0, x2=0, x1=1)  a=1
  index 2 = 0b010  (x3=0, x2=1, x1=0)  a=2
  index 3 = 0b011  (x3=0, x2=1, x1=1)  a=3
  index 4 = 0b100  (x3=1, x2=0, x1=0)  a=4
  index 5 = 0b101  (x3=1, x2=0, x1=1)  a=5
  index 6 = 0b110  (x3=1, x2=1, x1=0)  a=6
  index 7 = 0b111  (x3=1, x2=1, x1=1)  a=7


---
### 1.2 — Input: `q` (prime modulus)

| Property | Value |
|---|---|
| Type | `jnp.uint32` scalar |
| Value | A prime, at most `4_294_967_291` (the largest 32-bit prime = $2^{32} - 5$) |
| Role | All arithmetic is done in $\mathbb{F}_q$ — the finite field of size `q` |

**Why a prime?**  
In a finite field $\mathbb{F}_q$, every nonzero element has a multiplicative inverse. This makes the verifier's random challenge uniformly distributed over a field, which is what gives the protocol its soundness guarantees. With a composite modulus you'd have zero divisors and the math breaks.

**Why at most 32-bit?**  
Multiplication of two 32-bit values requires 64-bit intermediates. $(q-1)^2 \le (2^{32}-1)^2 < 2^{64}-1$, so `uint64` is just enough to hold the product before reduction.

In [3]:
# The two primes you'll see in the assignment
Q_small  = jnp.uint32(17)             # tiny — used in tutorial worked examples
Q_actual = jnp.uint32(4_294_967_291)  # 2^32 - 5 — used in real benchmark cases

print(f"Q_small  = {int(Q_small)}   dtype={Q_small.dtype}")
print(f"Q_actual = {int(Q_actual)}  dtype={Q_actual.dtype}")
print(f"Q_actual in binary bits: {int(Q_actual):b}  (fits in 32 bits: {int(Q_actual) < 2**32})")
print()
# Show overflow risk: multiplying two (q-1) values
max_val = int(Q_actual) - 1
product = max_val * max_val
print(f"(q-1)^2 = {product}")
print(f"Fits in uint32? {product < 2**32}   ← must promote to uint64 first")
print(f"Fits in uint64? {product < 2**64}   ← safe")

Q_small  = 17   dtype=uint32
Q_actual = 4294967291  dtype=uint32
Q_actual in binary bits: 11111111111111111111111111111011  (fits in 32 bits: True)

(q-1)^2 = 18446744022169944100
Fits in uint32? False   ← must promote to uint64 first
Fits in uint64? True   ← safe


---
### 1.3 — Input: `expression`

| Property | Value |
|---|---|
| Type | `list[list[str]]` |
| Outer list | Additive terms (joined by +) |
| Inner list | Multiplicative factors within one term (joined by ×) |
| Variable names | Must be from `('a','b','c','d','e','g')` |

#### All 7 expressions the assignment tests:

| Expression string | Python representation | Degree | eval points per round |
|---|---|---|---|
| `a` | `[['a']]` | 1 | 2: g(0), g(1) |
| `a*b` | `[['a','b']]` | 2 | 3: g(0), g(1), g(2) |
| `a*b + c` | `[['a','b'],['c']]` | 2 | 3: g(0), g(1), g(2) |
| `a*b*c` | `[['a','b','c']]` | 3 | 4: g(0), g(1), g(2), g(3) |
| `a*a*b*b*c` | `[['a','a','b','b','c']]` | 5 | 6: g(0)..g(5) |
| `a*b*c + d*e` | `[['a','b','c'],['d','e']]` | 3 | 4: g(0)..g(3) |
| `a*b*c*g + d*e*g` | `[['a','b','c','g'],['d','e','g']]` | 4 | 5: g(0)..g(4) |

**Degree rule:** `degree = sum(len(term) for term in expression)`  
The degree equals the maximum total number of variable factors in any **single evaluation** of the composed polynomial at one point.

In [4]:
# All expressions from provided.py
EXPRESSIONS = (
    [["a"]],
    [["a", "b"]],
    [["a", "b"], ["c"]],
    [["a", "b", "c"]],
    [["a", "a", "b", "b", "c"]],
    [["a", "b", "c"], ["d", "e"]],
    [["a", "b", "c", "g"], ["d", "e", "g"]],
)

print(f"{'Expression':30s}  {'Degree':6s}  {'Eval pts':8s}  {'Variables needed'}")
print("-" * 70)
for expr in EXPRESSIONS:
    degree    = sum(len(term) for term in expr)
    eval_pts  = degree + 1
    expr_str  = " + ".join("*".join(term) for term in expr)
    vars_used = sorted(set(v for term in expr for v in term))
    print(f"  {expr_str:28s}  {degree:6d}  {eval_pts:8d}  {vars_used}")

Expression                      Degree  Eval pts  Variables needed
----------------------------------------------------------------------
  a                                  1         2  ['a']
  a*b                                2         3  ['a', 'b']
  a*b + c                            3         4  ['a', 'b', 'c']
  a*b*c                              3         4  ['a', 'b', 'c']
  a*a*b*b*c                          5         6  ['a', 'b', 'c']
  a*b*c + d*e                        5         6  ['a', 'b', 'c', 'd', 'e']
  a*b*c*g + d*e*g                    7         8  ['a', 'b', 'c', 'd', 'e', 'g']


---
### 1.4 — Input: `challenges`

| Property | Value |
|---|---|
| Type | `jnp.ndarray` of `uint32` |
| Shape | `(num_rounds - 1,)` — **one fewer than the number of rounds** |
| Value range | Each element in `[1, q-1]` — sampled uniformly by the verifier |
| Why one fewer | The **last** challenge is the verifier's final-evaluation challenge. The prover does not need to apply it — the verifier uses it to check `g_n(r_n) == f(r1, r2, ..., rn)` directly. |

**Common mistake:** passing `num_rounds` challenges instead of `num_rounds - 1`. This causes the prover to apply an extra MLE fold on an already size-1 table, producing garbage.

```
num_rounds = 4
challenges  = [r1, r2, r3]    ← length 3 = num_rounds - 1  ✓
                r4 is the verifier-only final challenge
```

In [5]:
# Visualise the challenge timeline for a 4-round problem
num_rounds = 4
Q = jnp.uint32(17)

# Simulate verifier-generated challenges (all 4, but only 3 go to prover)
all_challenges = [8, 5, 11, 3]   # r1, r2, r3, r4

challenges = jnp.array(all_challenges[:-1], dtype=jnp.uint32)  # r1..r3
verifier_final = all_challenges[-1]                             # r4 — not passed to prover

print("Challenge timeline:")
print(f"  Total rounds:           {num_rounds}")
print(f"  All challenges:         {all_challenges}")
print(f"  Passed to sumcheck_32:  {challenges.tolist()}   shape={challenges.shape}  dtype={challenges.dtype}")
print(f"  Verifier-only (r{num_rounds}):   {verifier_final}   ← NOT in challenges array")
print()
print("Round-by-round prover actions:")
for round_idx in range(num_rounds):
    has_challenge = round_idx < len(challenges)
    if has_challenge:
        r = int(challenges[round_idx])
        print(f"  Round {round_idx+1}: compute g_{round_idx+1}(t)  →  apply MLE fold with r={r}")
    else:
        print(f"  Round {round_idx+1}: compute g_{round_idx+1}(t)  →  NO fold (last round; verifier holds r{num_rounds}={verifier_final})")

Challenge timeline:
  Total rounds:           4
  All challenges:         [8, 5, 11, 3]
  Passed to sumcheck_32:  [8, 5, 11]   shape=(3,)  dtype=uint32
  Verifier-only (r4):   3   ← NOT in challenges array

Round-by-round prover actions:
  Round 1: compute g_1(t)  →  apply MLE fold with r=8
  Round 2: compute g_2(t)  →  apply MLE fold with r=5
  Round 3: compute g_3(t)  →  apply MLE fold with r=11
  Round 4: compute g_4(t)  →  NO fold (last round; verifier holds r4=3)


---
### 1.5 — Outputs: `claim0` and `round_evals`

#### Output 1: `claim0`

| Property | Value |
|---|---|
| Type | `jnp.ndarray` scalar, dtype `uint32` |
| Shape | `()` — 0-dimensional scalar |
| Value | $S = \sum_{x \in \{0,1\}^n} f(x) \bmod q$ — the sum of `expression` over all Boolean inputs |
| When computed | **Before any MLE updates** — uses the full original tables |

#### Output 2: `round_evals`

| Property | Value |
|---|---|
| Type | `jnp.ndarray`, dtype `uint32` |
| Shape | `(num_rounds, degree+1)` — 2-D array |
| `round_evals[i]` | The round polynomial $g_{i+1}(0), g_{i+1}(1), \ldots, g_{i+1}(d)$ |
| `round_evals[i][t]` | The value of the univariate $g_{i+1}$ at point $t$ |

**Consistency invariant** the verifier checks:  
- `round_evals[0][0] + round_evals[0][1] ≡ claim0  (mod q)`  
- For `i > 0`: `round_evals[i][0] + round_evals[i][1] ≡ round_evals[i-1][r_i]  (mod q)`

In [6]:
# ── Full worked example matching the tutorial intro doc ─────────────────────
# f(x1,x2,x3) = x1 + 2*x2 + 4*x3  (mod 17)
# Table = [0,1,2,3,4,5,6,7]
# Challenges passed to prover: r1=8, r2=5   (r3=11 is verifier-only)

def mod_add_32(a, b, q):
    return ((a.astype(jnp.uint64) + b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

def mod_sub_32(a, b, q):
    q64 = jnp.uint64(q)
    return ((a.astype(jnp.uint64) + q64 - b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

def mod_mul_32(a, b, q):
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)

def mle_update_32(zero_eval, one_eval, target_eval, *, q):
    diff   = mod_sub_32(one_eval, zero_eval, q)
    scaled = mod_mul_32(diff, target_eval, q)
    return mod_add_32(scaled, zero_eval, q)

def eval_expression(tables, expression, q):
    total = None
    for term in expression:
        term_val = tables[term[0]]
        for var in term[1:]:
            term_val = mod_mul_32(term_val, tables[var], q)
        total = term_val if total is None else mod_add_32(total, term_val, q)
    return total

def sumcheck_32_ref(eval_tables, *, q, expression, challenges, num_rounds):
    degree          = sum(len(term) for term in expression)
    num_eval_points = degree + 1
    f_vals  = eval_expression(eval_tables, expression, q)
    claim0  = (jnp.sum(f_vals.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)
    tables  = dict(eval_tables)
    all_round_evals = []
    for round_idx in range(num_rounds):
        evens = {var: tbl[0::2] for var, tbl in tables.items()}
        odds  = {var: tbl[1::2] for var, tbl in tables.items()}
        round_evals = []
        for t_int in range(num_eval_points):
            t = jnp.array(t_int, dtype=jnp.uint32)
            if t_int == 0:
                tables_t = evens
            elif t_int == 1:
                tables_t = odds
            else:
                tables_t = {var: mle_update_32(evens[var], odds[var], t, q=q) for var in tables}
            vals = eval_expression(tables_t, expression, q)
            g_t  = (jnp.sum(vals.astype(jnp.uint64)) % jnp.uint64(q)).astype(jnp.uint32)
            round_evals.append(g_t)
        all_round_evals.append(jnp.stack(round_evals))
        if round_idx < len(challenges):
            r = challenges[round_idx]
            tables = {var: mle_update_32(evens[var], odds[var], r, q=q) for var in tables}
    return claim0, jnp.stack(all_round_evals)


# Run the example
Q          = jnp.uint32(17)
tables_ex  = {'a': jnp.array([0,1,2,3,4,5,6,7], dtype=jnp.uint32)}
chals_ex   = jnp.array([8, 5], dtype=jnp.uint32)  # length = num_rounds - 1 = 2
expr_ex    = [['a']]
n_rounds   = 3

claim0, round_evals = sumcheck_32_ref(
    tables_ex, q=Q, expression=expr_ex,
    challenges=chals_ex, num_rounds=n_rounds
)

print("OUTPUT SHAPES & DTYPES")
print("=" * 50)
print(f"  claim0       :  shape={claim0.shape}  dtype={claim0.dtype}  value={int(claim0)}")
print(f"  round_evals  :  shape={round_evals.shape}  dtype={round_evals.dtype}")
print()
print(f"  Expected claim0 = 11  →  {'PASS' if int(claim0)==11 else 'FAIL'}")
print()
print("ROUND EVALS (row i = round i+1):")
expected_re = [[12, 16], [3, 7], [1, 5]]
for i, (row, exp_row) in enumerate(zip(round_evals.tolist(), expected_re)):
    status = 'PASS' if row == exp_row else f'FAIL (expected {exp_row})'
    print(f"  round_evals[{i}] = {row}   ← g_{i+1}(0), g_{i+1}(1)   [{status}]")
print()
print("VERIFIER CONSISTENCY CHECKS:")
re = round_evals.tolist()
chals = [8, 5]
check0 = (re[0][0] + re[0][1]) % 17
print(f"  g1(0)+g1(1) = {re[0][0]}+{re[0][1]} = {check0} mod 17  == claim0({int(claim0)})?  {'YES' if check0==int(claim0) else 'NO'}")
for i in range(1, n_rounds):
    r = chals[i-1]
    lhs = (re[i][0] + re[i][1]) % 17
    rhs = re[i-1][r % len(re[i-1])] if r < len(re[i-1]) else 'need poly interp'
    print(f"  g{i+1}(0)+g{i+1}(1) = {re[i][0]}+{re[i][1]} = {lhs} mod 17  == g{i}(r{i}={r})={re[i-1]}[interp]")

OUTPUT SHAPES & DTYPES
  claim0       :  shape=()  dtype=uint32  value=11
  round_evals  :  shape=(3, 2)  dtype=uint32

  Expected claim0 = 11  →  PASS

ROUND EVALS (row i = round i+1):
  round_evals[0] = [12, 16]   ← g_1(0), g_1(1)   [PASS]
  round_evals[1] = [3, 7]   ← g_2(0), g_2(1)   [PASS]
  round_evals[2] = [1, 5]   ← g_3(0), g_3(1)   [PASS]

VERIFIER CONSISTENCY CHECKS:
  g1(0)+g1(1) = 12+16 = 11 mod 17  == claim0(11)?  YES
  g2(0)+g2(1) = 3+7 = 10 mod 17  == g1(r1=8)=[12, 16][interp]
  g3(0)+g3(1) = 1+5 = 6 mod 17  == g2(r2=5)=[3, 7][interp]


---
### 1.6 — I/O Summary Table

```
INPUTS
────────────────────────────────────────────────────────────────────────
eval_tables   dict[str → uint32[2^n]]   One table per variable; index=Boolean point
q             uint32 scalar             Prime modulus; all arithmetic is mod q
expression    list[list[str]]           Polynomial structure: additive terms of products
challenges    uint32[num_rounds-1]      Verifier's random challenges r1..r_{n-1}
num_rounds    int                       n = number of Boolean variables

OUTPUTS
────────────────────────────────────────────────────────────────────────
claim0        uint32 scalar ()          Initial sum S = Σ_{x∈{0,1}^n} f(x) mod q
round_evals   uint32[num_rounds, d+1]   Per-round polynomial: row i = [g_{i+1}(0)..g_{i+1}(d)]

WHERE:
  n  = num_rounds
  d  = degree = sum(len(term) for term in expression)
  d+1 = number of evaluation points per round polynomial
```

---
## Section 2 — Operation Taxonomy: Every Op Mapped to Hardware Primitives

Understanding *what kind* of operation each step is determines whether it benefits from GPU/TPU acceleration.  
We use 4 categories:

| Category | Notation | Description | Example |
|---|---|---|---|
| **Vec ⊙ Vec** | `V[N] op V[N] → V[N]` | Elementwise op on two arrays of the same shape | `a + b` pointwise |
| **Vec ⊙ Scalar** | `V[N] op s → V[N]` | Broadcast a single scalar across an array | `table * r` where `r` is a challenge |
| **Reduction** | `V[N] → s` | Collapse an array to a scalar | `sum(table)` |
| **Gather/Stride** | `V[N] → V[N/2]` | Non-unit stride access | `table[0::2]` (even elements) |
| **Sequential loop** | `s → s` (repeated) | One iteration depends on previous | outer round loop |

**Matmuls do not appear in SumCheck.** This matters: GPUs are specifically designed for matmul, but SumCheck is dominated by elementwise ops and reductions — a different compute pattern.

---
### 2.1 — Operation Map: Every Step in `sumcheck_32`

```
STEP 0 — Compute claim0
────────────────────────────────────────────────────────────────────────
  a) eval_expression(tables, expr, q)
       for each term [v1,v2,...] in expression:
         term_val = tables[v1]                        ← identity (no op)
         term_val = mod_mul_32(term_val, tables[v2])  ← VEC ⊙ VEC  (uint64 intermediate)
         term_val = mod_mul_32(term_val, tables[v3])  ← VEC ⊙ VEC
         ...repeat for each factor...
         total    = mod_add_32(total, term_val)        ← VEC ⊙ VEC
       Array size throughout: (2^n,)

  b) claim0 = sum(f_vals.astype(uint64)) % q          ← REDUCTION  V[2^n] → scalar

STEP 1 — Per-round: split table
────────────────────────────────────────────────────────────────────────
  evens = table[0::2]    ← GATHER/STRIDE  V[2^k] → V[2^(k-1)]  (even elements)
  odds  = table[1::2]    ← GATHER/STRIDE  V[2^k] → V[2^(k-1)]  (odd elements)

STEP 2 — Per-round: compute g(t) for t = 0, 1, ..., degree
────────────────────────────────────────────────────────────────────────
  For t=0:  tables_t = evens             ← no computation (pointer alias)
  For t=1:  tables_t = odds              ← no computation (pointer alias)
  For t≥2:  mle_update_32(evens, odds, t, q)
              diff   = mod_sub_32(odds, evens, q)    ← VEC ⊙ VEC
              scaled = mod_mul_32(diff,  t,    q)    ← VEC ⊙ SCALAR  (t is a small integer)
              result = mod_add_32(scaled, evens, q)  ← VEC ⊙ VEC

  Then eval_expression + sum (same as Step 0b above, on half-size arrays)
            eval_expression                           ← VEC ⊙ VEC  (size 2^(k-1))
            sum(vals.astype(uint64)) % q              ← REDUCTION  V[2^(k-1)] → scalar

  Outer loop: t = 0..degree  ← SEQUENTIAL (degree is 1–5, tiny; effectively unrolled)

STEP 3 — Per-round: apply MLE update with challenge r
────────────────────────────────────────────────────────────────────────
  mle_update_32(evens, odds, r, q)  (r = challenges[round_idx])
    diff   = mod_sub_32(odds, evens, q)    ← VEC ⊙ VEC   (size 2^(k-1))
    scaled = mod_mul_32(diff, r, q)         ← VEC ⊙ SCALAR
    result = mod_add_32(scaled, evens, q)   ← VEC ⊙ VEC
  New table size: 2^(k-1)  (table halves each round)

STEP 4 — Outer round loop
────────────────────────────────────────────────────────────────────────
  for round_idx in range(num_rounds):    ← SEQUENTIAL (cannot parallelize)
    Each round depends on the previous round's challenge and table state.
```

In [7]:
# ── Demonstrate each operation category in isolation ──────────────────────
Q  = jnp.uint32(17)
N  = 8   # 2^3 table

a_tbl = jnp.array([0,1,2,3,4,5,6,7], dtype=jnp.uint32)
b_tbl = jnp.array([1,0,1,0,1,0,1,0], dtype=jnp.uint32)
r     = jnp.uint32(8)   # a challenge scalar

print("Operation taxonomy demonstration")
print("=" * 60)

# VEC ⊙ VEC: mod_mul_32 on two tables
prod = mod_mul_32(a_tbl, b_tbl, Q)
print(f"VEC ⊙ VEC   mod_mul_32(a, b, q)")
print(f"  a    : {a_tbl.tolist()}")
print(f"  b    : {b_tbl.tolist()}")
print(f"  a*b  : {prod.tolist()}   ← same shape {prod.shape}, each element independent")
print()

# VEC ⊙ SCALAR: mod_mul_32 with a scalar challenge
scaled_a = mod_mul_32(a_tbl, r, Q)
print(f"VEC ⊙ SCALAR  mod_mul_32(a, r=8, q)")
print(f"  a    : {a_tbl.tolist()}")
print(f"  r    : {int(r)}  ← single scalar, broadcast across all elements")
print(f"  a*r  : {scaled_a.tolist()}")
print()

# REDUCTION: sum → scalar
total = int(jnp.sum(a_tbl.astype(jnp.uint64)) % jnp.uint64(Q))
print(f"REDUCTION     sum(a) mod q")
print(f"  a    : {a_tbl.tolist()}")
print(f"  sum  : {total}   ← scalar, shape ()")
print()

# GATHER/STRIDE: even/odd split
evens = a_tbl[0::2]
odds  = a_tbl[1::2]
print(f"GATHER/STRIDE  table[0::2] and table[1::2]")
print(f"  a      : {a_tbl.tolist()}   shape {a_tbl.shape}")
print(f"  evens  : {evens.tolist()}          shape {evens.shape}  (x1=0 entries)")
print(f"  odds   : {odds.tolist()}          shape {odds.shape}  (x1=1 entries)")
print()

# MLE UPDATE: Vec ⊙ Vec + Vec ⊙ Scalar chain
folded = mle_update_32(evens, odds, r, q=Q)
print(f"MLE UPDATE    mle_update_32(evens, odds, r=8, q)")
print(f"  evens  : {evens.tolist()}")
print(f"  odds   : {odds.tolist()}")
print(f"  folded : {folded.tolist()}")
print(f"  Operations: (odds-evens)=VEC⊙VEC → *r=VEC⊙SCALAR → +evens=VEC⊙VEC")

Operation taxonomy demonstration
VEC ⊙ VEC   mod_mul_32(a, b, q)
  a    : [0, 1, 2, 3, 4, 5, 6, 7]
  b    : [1, 0, 1, 0, 1, 0, 1, 0]
  a*b  : [0, 0, 2, 0, 4, 0, 6, 0]   ← same shape (8,), each element independent

VEC ⊙ SCALAR  mod_mul_32(a, r=8, q)
  a    : [0, 1, 2, 3, 4, 5, 6, 7]
  r    : 8  ← single scalar, broadcast across all elements
  a*r  : [0, 8, 16, 7, 15, 6, 14, 5]

REDUCTION     sum(a) mod q
  a    : [0, 1, 2, 3, 4, 5, 6, 7]
  sum  : 11   ← scalar, shape ()

GATHER/STRIDE  table[0::2] and table[1::2]
  a      : [0, 1, 2, 3, 4, 5, 6, 7]   shape (8,)
  evens  : [0, 2, 4, 6]          shape (4,)  (x1=0 entries)
  odds   : [1, 3, 5, 7]          shape (4,)  (x1=1 entries)

MLE UPDATE    mle_update_32(evens, odds, r=8, q)
  evens  : [0, 2, 4, 6]
  odds   : [1, 3, 5, 7]
  folded : [8, 10, 12, 14]
  Operations: (odds-evens)=VEC⊙VEC → *r=VEC⊙SCALAR → +evens=VEC⊙VEC


---
### 2.2 — Operation Count as a Function of `n` and `degree`

Let $n$ = `num_rounds`, $N = 2^n$, $d$ = degree, $V$ = number of distinct variable tables.

| Step | Operation type | Count (elementwise ops on arrays) |
|---|---|---|
| claim0: eval expression | VEC ⊙ VEC mul | $(V-1) \cdot N$ |
| claim0: sum | Reduction | $N$ |
| Per round, split | Gather ×2 | $N_k$ (current table size) |
| Per round, g(t) for $t\ge 2$ | VEC ⊙ VEC + VEC ⊙ SCALAR | $\sim 3 \cdot (d-1) \cdot N_k/2$ per round |
| Per round, eval expression on half | VEC ⊙ VEC mul | $(V-1) \cdot N_k/2$ per eval point |
| Per round, $d+1$ reductions | Reduction | $(d+1) \cdot N_k/2$ per round |
| Per round, MLE fold | 3× VEC ⊙ VEC/SCALAR | $3 \cdot N_k/2$ |

**Table size halves every round**: $N_0=N, N_1=N/2, N_2=N/4, \ldots$  
Total elementwise work across all rounds is **geometric series** $\approx 2N$ — the first round dominates.

This means: **GPU parallelism matters most in round 1** (largest table), and is progressively less important in later rounds.

In [8]:
# Count elementwise operations per round for a concrete example
def count_ops(num_rounds, expression):
    """Count approximate elementwise ops at each round."""
    degree = sum(len(term) for term in expression)
    V = len({v for term in expression for v in term})
    n_eval_pts = degree + 1

    rows = []
    N_k = 2**num_rounds
    for rnd in range(num_rounds):
        half = N_k // 2
        # ops for computing g(t) at all evaluation points
        mle_ops   = (degree - 1) * 3 * half   # MLE extrapolation for t>=2
        expr_ops  = (V - 1) * half * n_eval_pts  # expression eval (mul chains)
        reduc_ops = half * n_eval_pts              # reductions (sum)
        fold_ops  = 3 * half                       # MLE fold with challenge
        total     = mle_ops + expr_ops + reduc_ops + fold_ops
        rows.append((rnd+1, N_k, half, total))
        N_k = half
    return rows

expr  = [['a','b','c']]  # a*b*c  degree=3, 3 vars
n     = 6

print(f"Operation counts for n={n}, expr=a*b*c (degree=3)")
print(f"{'Round':6s}  {'Table size':12s}  {'Half size':10s}  {'~Elem ops':12s}  {'% of total'}")
print("-" * 60)
ops_per_round = count_ops(n, expr)
grand_total   = sum(r[3] for r in ops_per_round)
for rnd, N_k, half, total in ops_per_round:
    pct = 100 * total / grand_total
    print(f"  {rnd:4d}  {N_k:12,}  {half:10,}  {total:12,}  {pct:6.1f}%")
print("-" * 60)
print(f"  {'TOTAL':4s}  {'':12s}  {'':10s}  {grand_total:12,}  100.0%")
print()
print(f"Round 1 alone accounts for {100*ops_per_round[0][3]/grand_total:.1f}% of all elementwise work.")

Operation counts for n=6, expr=a*b*c (degree=3)
Round   Table size    Half size   ~Elem ops     % of total
------------------------------------------------------------
     1            64          32           672    50.8%
     2            32          16           336    25.4%
     3            16           8           168    12.7%
     4             8           4            84     6.3%
     5             4           2            42     3.2%
     6             2           1            21     1.6%
------------------------------------------------------------
  TOTAL                                   1,323  100.0%

Round 1 alone accounts for 50.8% of all elementwise work.


---
### 2.3 — Why There Are No Matrix Multiplications

GPUs and TPUs are **especially** fast at matrix multiplications (GEMM) because they have dedicated hardware units (tensor cores on NVIDIA, MXUs on TPUs) that can compute a large matrix product in a single fused operation.

SumCheck does **not use matmul** for two reasons:

1. **The evaluation tables are 1-D vectors**, not matrices. There is no structured 2-D computation pattern.
2. **Each table entry is independent** — the combination rule is elementwise multiplication of the tables, not a dot product between rows and columns.

The closest structure to a matmul that appears is in the `vmap` optimisation (Section 8.4 of the tutorial), where you can think of computing `g(0), g(1), ..., g(d)` simultaneously as a small matrix of shape `(d+1, N/2)`. But `d` is at most 5, so this is not a meaningful matmul workload.

**Implication:** SumCheck is **memory-bandwidth bound**, not compute bound. The bottleneck is loading and storing the `2^n`-element tables from GPU HBM (high-bandwidth memory), not the arithmetic itself.

---
## Section 3 — CPU vs GPU/TPU: How the Hardware Executes Each Operation

This section explains *concretely* what happens inside the hardware for each operation category, and why the performance gap is large.

---
### 3.1 — How a CPU Executes SumCheck Today

A modern CPU (e.g., Apple M2 or Intel Xeon) runs SumCheck as follows:

#### Step-by-step CPU execution of `mod_mul_32(a, b, q)` on a 2^20 array:

```
CPU SIMD (vectorized, 1 core):
  Register width: 256-bit AVX2  →  holds 4× uint64 values
  Per SIMD instruction: compute 4 elements in parallel
  Total iterations = 2^20 / 4 = 262,144 SIMD instructions

With 8 cores (OpenMP or JAX default):
  Effective throughput ≈ 8 × 4 = 32 elements per cycle
  At 3 GHz: ~96 billion elements/second theoretical
  Real throughput (memory-bound): ~3–10 Gelem/s
```

#### CPU execution of the round loop:

```
for round_idx in range(n):          ← Python interpreter runs this
    evens = table[0::2]             ← memory copy (stride-2 gather)
    odds  = table[1::2]             ← memory copy (stride-2 gather)
    for t in range(degree+1):       ← Python interpreter runs this
        ...compute g(t)...          ← JAX dispatches to XLA CPU kernel
    fold tables with challenge r    ← JAX dispatches to XLA CPU kernel
```

**Key CPU bottlenecks:**
1. **Python loop overhead**: The `for round_idx` and `for t` loops run in the Python interpreter — each iteration pays ~microseconds of Python overhead.
2. **Memory bandwidth**: For `n=20` (2^20 entries × 4 bytes = 4 MB per table), the data barely fits in L3 cache. Loading and storing it multiple times per round saturates memory bandwidth.
3. **Sequential rounds**: All `n` rounds run one after another on the same cores.

---
### 3.2 — How a GPU Executes SumCheck

A GPU (e.g., NVIDIA A100) has a radically different architecture:

```
A100 specs (relevant to SumCheck):
  Streaming Multiprocessors (SMs): 108
  CUDA cores per SM: 64
  Total CUDA cores: 6,912
  Clock speed: ~1.4 GHz
  HBM2e memory bandwidth: 2 TB/s
  L2 cache: 40 MB
```

#### GPU execution of `mod_mul_32(a, b, q)` on a 2^20 array:

```
GPU CUDA kernel launch:
  Thread blocks of 256 threads, each computing 1 element
  Total threads launched: 2^20 = 1,048,576
  Thread blocks: 1,048,576 / 256 = 4,096 blocks
  All 6,912 CUDA cores work in parallel
  At 1.4 GHz: ~9.7 GFLOP/s of int64 ops
  Real throughput: ~50–200 Gelem/s (memory-bandwidth limited)
```

**The key difference:** on a CPU, 32 elements run in parallel. On a GPU, **6,912 elements** run in parallel — a ~200× increase in parallelism for elementwise ops.

#### GPU execution of the MLE update chain:

With `jax.jit`, XLA **fuses** the three operations into a single kernel:
```
Without JIT:
  Kernel 1: diff   = odds - evens  → write to HBM
  Kernel 2: scaled = diff * r      → read from HBM, write to HBM
  Kernel 3: result = scaled + evens → read from HBM, write to HBM
  Total HBM traffic: 6 reads + 3 writes = 9 × N × 4 bytes

With JIT (kernel fusion):
  Single fused kernel:
    read odds, read evens  → compute diff, scaled, result in registers → write result
  Total HBM traffic: 2 reads + 1 write = 3 × N × 4 bytes
  Speedup from fusion: 3× reduction in memory traffic
```

---
### 3.3 — How a TPU Executes SumCheck

A TPU (Tensor Processing Unit) is optimized for matrix multiplications and is less naturally suited to SumCheck:

```
TPU v4 specs:
  Matrix Multiply Units (MXUs): 4
  MXU systolic array: 128×128 bfloat16
  Peak throughput: 275 TFLOPS (but for matmul only)
  HBM bandwidth: 1.2 TB/s
```

**SumCheck on TPU:**
- The MXUs are **idle** — there are no matrix multiplications.
- Elementwise ops run on the TPU's scalar cores (much fewer than GPU CUDA cores).
- Advantage: XLA on TPU is **very aggressive at operator fusion** — the entire round's computation (split, MLE update, expression eval, sum) may fuse into one operation.
- The `vmap` trick (Section 8.4 of the tutorial) helps TPU more than GPU because it creates a batched operation that XLA can better schedule.

**Bottom line:** TPU is not the best hardware for SumCheck. GPU wins for this workload because its CUDA cores handle elementwise integer ops natively.

---
### 3.4 — Operation-Level CPU vs GPU Comparison

```
OPERATION          CPU (8 cores, AVX2)        GPU (A100)              Speedup
────────────────────────────────────────────────────────────────────────────────
mod_mul_32         ~32 elem/cycle             ~6912 elem/cycle        ~216×
N=2^20 table       saturates L3 cache         fits in L2 cache (40MB) better locality

mod_add_32         ~32 elem/cycle             ~6912 elem/cycle        ~216×

table[0::2]        stride-2 gather            coalesced → stride-2    similar
(even/odd split)   (suboptimal prefetcher)    (good memory access)    

jnp.sum            parallel reduce, 8 trees   parallel reduce, 6912   ~864×
(claim0 / g(t))    cores → sequential merge   threads → O(log N) tree  

round loop         Python sequential          Python sequential        1× (same)
(n iterations)     ~n×(Python overhead)       ~n×(kernel launch)      GPU overhead≈CPU

degree loop        Python sequential          Python sequential        1×
(d+1 iters, d≤5)   tiny; often faster on CPU  kernel launch overhead  CPU may win
```

**The pattern is clear:**
- For the table-level ops (which dominate by $>95\%$ of compute), GPU is $\sim 100$–$200\times$ faster.
- For the sequential Python loops, both are equally slow — the GPU doesn't help at all.
- `jax.jit` eliminates Python overhead from inside the fused kernel, but not from the `for round_idx` and `for t` outer loops.

This is exactly the memory-bandwidth / control-flow separation the papers (NoCap, zkSpeed, zkPHIRE) are trying to close with custom hardware.

In [ ]:
import time

Q_big = jnp.uint32(4_294_967_291)
key   = jax.random.PRNGKey(42)
N     = 2**20  # 1M elements

a_big = jax.random.randint(key, (N,), 0, int(Q_big), dtype=jnp.uint32)
b_big = jax.random.randint(key, (N,), 0, int(Q_big), dtype=jnp.uint32)

jit_mul = jax.jit(mod_mul_32)
jit_add = jax.jit(mod_add_32)
jit_sum = jax.jit(lambda x: jnp.sum(x.astype(jnp.uint64)) % jnp.uint64(Q_big))

# Warm up JIT
_ = jit_mul(a_big, b_big, Q_big).block_until_ready()
_ = jit_add(a_big, b_big, Q_big).block_until_ready()
_ = jit_sum(a_big).block_until_ready()

RUNS = 20

def bench(fn, *args, label):
    t0 = time.perf_counter()
    for _ in range(RUNS):
        fn(*args).block_until_ready()
    ms = (time.perf_counter() - t0) / RUNS * 1000
    gelem = N / (ms / 1000) / 1e9
    print(f"  {label:25s}  {ms:7.3f} ms/call   {gelem:.2f} Gelem/s")

print(f"Benchmark: N=2^20={N:,} elements")
print(f"Backend: {jax.default_backend()}")
print("=" * 65)
bench(jit_mul, a_big, b_big, Q_big, label="mod_mul_32 (VEC⊙VEC)")
bench(jit_add, a_big, b_big, Q_big, label="mod_add_32 (VEC⊙VEC)")
bench(jit_sum, a_big,         label="sum (REDUCTION)")

# MLE update — q is a regular dynamic argument, not static
evens_big = a_big[0::2]
odds_big  = a_big[1::2]
r_big     = jnp.uint32(123456789)

# q is a JAX array so it must be a traced (dynamic) arg, not a static one
jit_mle = jax.jit(lambda z, o, t, q: mle_update_32(z, o, t, q=q))
_ = jit_mle(evens_big, odds_big, r_big, Q_big).block_until_ready()

t0 = time.perf_counter()
for _ in range(RUNS):
    jit_mle(evens_big, odds_big, r_big, Q_big).block_until_ready()
ms = (time.perf_counter() - t0) / RUNS * 1000
gelem = (N // 2) / (ms / 1000) / 1e9
print(f"  {'mle_update_32 (fused)':25s}  {ms:7.3f} ms/call   {gelem:.2f} Gelem/s")

---
### 3.5 — Memory Hierarchy: Where the Data Lives

Understanding the memory hierarchy explains why `jax.jit` (kernel fusion) matters so much.

```
CPU MEMORY HIERARCHY (approximate latency + bandwidth)
──────────────────────────────────────────────────────
L1 cache   32 KB      4 cycles    2 TB/s per core
L2 cache   256 KB     12 cycles   1 TB/s per core
L3 cache   32 MB      40 cycles   200 GB/s shared
DRAM       unlimited  200 cycles  50 GB/s shared

2^20 table × 4 bytes = 4 MB  →  fits in L3, tight
2^20 table × 3 vars  = 12 MB →  exceeds L3, goes to DRAM

GPU MEMORY HIERARCHY (A100)
──────────────────────────────────────────────────────
Registers  per thread  0 cycles    ~infinity
Shared mem 164 KB/SM   ~32 cycles  19 TB/s
L2 cache   40 MB       ~200 cycles 12 TB/s
HBM (DRAM) 80 GB       ~500 cycles 2 TB/s

2^20 table × 4 bytes = 4 MB  →  fits in GPU L2 (40 MB)
2^20 table × 3 vars  = 12 MB →  fits in GPU L2
2^24 table × 6 vars  = 384 MB → goes to HBM, still 2 TB/s
```

**Key insight:** For `n ≤ 20`, the GPU L2 cache (40 MB) holds the entire working set. The CPU L3 (32 MB) does not. This gives GPU an extra advantage beyond raw parallelism — **better cache hit rate**.

**Kernel fusion (jax.jit) saves memory traffic:**
```
Unfused MLE update: 3 kernels → 9 HBM reads+writes  (at 2 TB/s: ~some ns/elem)
Fused   MLE update: 1 kernel  → 3 HBM reads+writes  (3× faster)
```

---
### 3.6 — The Irreducible Sequential Part: Amdahl's Law in Action

Amdahl's Law says: if fraction $f$ of your program is sequential, the maximum speedup from parallelism is $1/(f)$.

In SumCheck:
- The **outer round loop** (`for round_idx in range(n)`) is **fully sequential** — each round's table depends on the previous round's challenge.
- The **inner degree loop** (`for t in range(degree+1)`) is sequential in Python (though it can be vmapped).

For `n=20` rounds, even if each round's table ops run **infinitely fast** on GPU, you still pay `n=20` Python loop iterations with kernel launch overhead.

```
Execution timeline for n=20, degree=3:

Round 1:  [GPU: 2^20 elems in parallel]  → Python overhead →
Round 2:  [GPU: 2^19 elems in parallel]  → Python overhead →
Round 3:  [GPU: 2^18 elems in parallel]  → Python overhead →
...                                                            
Round 20: [GPU: 2^1  elems in parallel]  → Python overhead → done

The Python overhead (kernel launch, Python interpreter) is ~10–100 μs per round.
For n=20: 20 × 100 μs = 2 ms of unavoidable sequential overhead.
GPU ops for n=20, degree=3: ~5–10 ms total.
Sequential fraction: ~20–30% of total time.
```

**This is why `jax.jit` on the whole round** (as shown in Section 8.4–8.6 of the tutorial) matters: it moves the computation inside a single traced XLA program, eliminating the Python-level round loop overhead.

In [ ]:
# Simulate the Amdahl's Law tradeoff for different n values
import math

print("Amdahl's Law estimate for SumCheck")
print("Assumptions: GPU makes table ops 100× faster than CPU; sequential part unchanged")
print()
print(f"{'n':4s}  {'N=2^n':10s}  {'Table ops (ms)':16s}  {'Sequential (ms)':17s}  {'GPU speedup':12s}")
print("-" * 70)

# Rough model:
#   CPU table ops time ∝ N (at ~1 Gelem/s), across all rounds → ~2N total
#   GPU table ops time ∝ N / 100 (100× speedup)
#   Sequential overhead: ~n × 0.05 ms (Python + kernel launch)

for n in [4, 8, 12, 16, 20, 24]:
    N = 2**n
    cpu_table_ms  = 2 * N / 1e9 * 1000  # 1 Gelem/s, ~2N total ops
    gpu_table_ms  = cpu_table_ms / 100
    seq_ms        = n * 0.05             # 50 μs per round (Python overhead)
    gpu_total_ms  = gpu_table_ms + seq_ms
    cpu_total_ms  = cpu_table_ms + seq_ms
    speedup       = cpu_total_ms / gpu_total_ms
    print(f"  {n:2d}  {N:10,}  {cpu_table_ms:8.2f} ms (CPU)  {seq_ms:8.2f} ms          {speedup:8.1f}×")

print()
print("At n=4:  sequential overhead dominates → low speedup")
print("At n=20: table ops dominate → near 100× speedup from GPU")
print("At n=24: approaches theoretical max speedup")

---
### 3.7 — Summary: What the GPU/TPU Buys You

```
┌──────────────────────────────────────────────────────────────────────────────┐
│            OPERATION          CPU         GPU (JIT)   Why GPU Wins           │
├──────────────────────────────────────────────────────────────────────────────┤
│ mod_mul_32 on 2^20 table     ~20 ms      ~0.2 ms     6912 vs 32 parallel     │
│ mod_add_32 on 2^20 table     ~10 ms      ~0.1 ms     same                    │
│ MLE update (fused, JIT)      ~30 ms      ~0.3 ms     fusion + parallelism    │
│ jnp.sum (claim0)             ~5 ms       ~0.1 ms     parallel tree reduce    │
│ Table split (even/odd)       ~5 ms       ~0.05 ms    coalesced memory        │
│ Round loop (n iterations)    ~1 ms       ~1 ms       sequential; no win      │
│ Degree loop (d+1 iters)      ~0.1 ms     ~0.5 ms     kernel launch overhead  │
├──────────────────────────────────────────────────────────────────────────────┤
│ Overall n=20 sumcheck        ~200 ms     ~3–5 ms     ~50–100× real speedup   │
└──────────────────────────────────────────────────────────────────────────────┘

The three things that matter most for GPU performance:
  1. jax.jit  →  kernel fusion, eliminate Python overhead from inner ops
  2. uint64 intermediates  →  correct results without overflow
  3. Large n (n≥16)  →  table-op time >> sequential overhead

The one thing that limits GPU performance:
  The n-round sequential loop — unavoidable by design of the SumCheck protocol.
```

In [ ]:
# ── Full end-to-end benchmark across all 7 expressions ────────────────────
import time

Q_bench = jnp.uint32(4_294_967_291)
n_bench = 16   # adjust to 20 if you have enough memory
WARMUP, RUNS = 3, 8

EXPRESSIONS = (
    [["a"]],
    [["a", "b"]],
    [["a", "b"], ["c"]],
    [["a", "b", "c"]],
    [["a", "a", "b", "b", "c"]],
    [["a", "b", "c"], ["d", "e"]],
    [["a", "b", "c", "g"], ["d", "e", "g"]],
)

key = jax.random.PRNGKey(0)
N   = 2**n_bench

print(f"Backend: {jax.default_backend()}   n={n_bench}   N=2^{n_bench}={N:,}")
print(f"{'Expression':30s}  {'Degree':6s}  {'Mean ms':8s}  {'ms/round'}")
print("-" * 65)

for expr in EXPRESSIONS:
    degree    = sum(len(term) for term in expr)
    expr_str  = " + ".join("*".join(t) for t in expr)
    vars_used = sorted(set(v for term in expr for v in term))

    tables = {v: jax.random.randint(key, (N,), 0, int(Q_bench), dtype=jnp.uint32)
              for v in vars_used}
    chals  = jax.random.randint(key, (n_bench-1,), 1, int(Q_bench), dtype=jnp.uint32)

    for _ in range(WARMUP):
        c0, re = sumcheck_32_ref(tables, q=Q_bench, expression=expr,
                                 challenges=chals, num_rounds=n_bench)
        re.block_until_ready()

    t0 = time.perf_counter()
    for _ in range(RUNS):
        c0, re = sumcheck_32_ref(tables, q=Q_bench, expression=expr,
                                 challenges=chals, num_rounds=n_bench)
        re.block_until_ready()
    mean_ms = (time.perf_counter() - t0) / RUNS * 1000

    print(f"  {expr_str:28s}  {degree:6d}  {mean_ms:8.2f}  {mean_ms/n_bench:.2f}")

---
## Final Reference Card

### I/O at a glance
```python
# INPUTS
eval_tables  : dict[str, uint32[2^n]]   # tables['a'][i] = a evaluated at boolean point i
q            : uint32 scalar            # prime modulus (≤ 4_294_967_291)
expression   : list[list[str]]          # [['a','b'],['c']] means a*b + c
challenges   : uint32[n-1]             # verifier's random challenges r1..r_{n-1}
num_rounds   : int                     # n = number of Boolean variables

# OUTPUTS
claim0       : uint32 scalar ()        # S = Σ f(x) mod q over all x in {0,1}^n
round_evals  : uint32[n, d+1]          # round_evals[i] = [g_{i+1}(0),...,g_{i+1}(d)]
```

### Operation taxonomy
```
mod_mul_32(V, V, q)     →  VEC ⊙ VEC    (dominant op, fully parallel)
mod_mul_32(V, s, q)     →  VEC ⊙ SCALAR (MLE extrapolation; s = eval point t)
mod_mul_32(V, r, q)     →  VEC ⊙ SCALAR (MLE fold; r = challenge)
mod_add_32(V, V, q)     →  VEC ⊙ VEC    (expression accumulation)
mod_sub_32(V, V, q)     →  VEC ⊙ VEC    (MLE diff computation)
jnp.sum(V) % q          →  REDUCTION    (claim0 and each g(t))
table[0::2], table[1::2] → GATHER       (even/odd split; table halves)
for t in range(d+1)     →  SEQUENTIAL   (tiny; d≤5; vmappable)
for round in range(n)   →  SEQUENTIAL   (unavoidable; each round needs prev challenge)
```

### GPU wins because
1. **6912 vs 32 parallel units** — elementwise ops run ~200× faster
2. **40 MB L2 cache** — entire working set for n≤20 stays on-chip
3. **jax.jit kernel fusion** — MLE update becomes 1 kernel instead of 3; 3× less memory traffic
4. **Parallel reduction** — `jnp.sum` on 2^20 elements takes O(log N) steps with 6912 threads

### GPU is bounded by
1. **Sequential outer loop** (n rounds, unavoidable by protocol design)
2. **HBM bandwidth** for n≥20 (data too large for L2; limited to 2 TB/s)
3. **Kernel launch overhead** for the inner degree loop (solved by vmap or static unrolling)